# Candidate LFC heatmap — plotting workbench

Extracted from the `candidate_LFC_heatmap` section of `plot_candidates()` in [development_pathway_analysis.py](development_pathway_analysis.py).

Run the cells from top to bottom. This notebook contains the plotting code and loads the existing exported plot inputs; it does not import or run the analysis script. Dependencies: NumPy, pandas, Matplotlib, and seaborn (the project's existing Python environment).

The default plot uses the original displayed genes, row order, target palette, color limits, and significance mask. Full matrices remain available for selecting a different subset. Rows belong to one display group for layout; `target_groups` retains overlapping memberships. Stars use the saved significance calls (finite p-value, adjusted p-value and LFC, padj below the saved threshold, and absolute LFC at least the saved threshold). Grey cells are missing values. Colors clip the display only.

The supplied DE source labels do not establish biological numerator/reference orientation. Keep that qualification when relabeling contrasts.

**Editing workflow:** load data once, adjust the appearance cell, then rerun the plotting cell. Export is optional and disabled by default.

In [ ]:
from pathlib import Path
import json
from unittest import async_case

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Locate the project from either its root or scripts/ (or a deeper directory).
ROOT = next(
    (path for path in [Path.cwd(), *Path.cwd().parents]
     if (path / "scripts/development_pathway_analysis.py").is_file()),
    None,
)
if ROOT is None:
    raise FileNotFoundError("Open this notebook from within the hydra_transcriptome_analysis project.")

DATA_DIR = ROOT / "results/processedData/development_pathways"
# Optional exports go into a separate folder, leaving the original figure intact.
EXPORT_DIR = ROOT / "results/figures/development_pathways/heatmap_refactor"

## Load only the saved heatmap inputs

All files below are under `results/processedData/development_pathways/`.

| File | Purpose |
| --- | --- |
| `candidate_LFC_matrix.tsv` | Full numeric gene × contrast LFC matrix; blanks stay missing. |
| `candidate_significance_matrix.tsv` | Boolean flags for the star overlay, aligned by gene ID and contrast. |
| `displayed_LFC_heatmap_rows.tsv` | Original displayed subset, exact row order, display group and gene annotations. |
| `candidate_rules.tsv` | Original target order, including targets absent from the displayed subset; preserves palette assignments. |
| `configuration_used.json` | Original clipping limit and significance thresholds. |

These exports must already exist from the development pathway analysis. No raw counts, annotation matching, DE fitting, or unrelated plots are needed. The saved significance flags do not change when you adjust the color limit.

In [ ]:
required_files = [
    "candidate_LFC_matrix.tsv",
    "candidate_significance_matrix.tsv",
    "displayed_LFC_heatmap_rows.tsv",
    "candidate_rules.tsv",
    "configuration_used.json",
]
missing = [name for name in required_files if not (DATA_DIR / name).is_file()]
if missing:
    raise FileNotFoundError(f"Missing saved heatmap inputs in {DATA_DIR}: {missing}")

cfg = json.loads((DATA_DIR / "configuration_used.json").read_text(encoding="utf-8"))
lfc = pd.read_csv(DATA_DIR / "candidate_LFC_matrix.tsv", sep="\t", index_col="query")
sig = pd.read_csv(DATA_DIR / "candidate_significance_matrix.tsv", sep="\t", index_col="query")
shown_original = pd.read_csv(DATA_DIR / "displayed_LFC_heatmap_rows.tsv", sep="\t", keep_default_na=False)
rules = pd.read_csv(DATA_DIR / "candidate_rules.tsv", sep="\t", keep_default_na=False, usecols=["target_group"])
target_order = rules["target_group"].drop_duplicates().tolist()

# Fail visibly if exports cannot be aligned or boolean flags were not parsed.
if not lfc.index.is_unique or not sig.index.is_unique or not shown_original["query"].is_unique:
    raise ValueError("Gene IDs must be unique in the matrices and displayed rows.")
if not sig.dtypes.eq(bool).all():
    raise ValueError("Significance matrix must contain True/False flags without missing values.")
if set(lfc.columns) != set(sig.columns):
    raise ValueError("LFC and significance matrices have different contrasts.")
if not lfc.index.equals(sig.index):
    raise ValueError("LFC and significance matrices have different gene IDs or row order.")
if shown_original.empty:
    raise ValueError("The saved heatmap has no displayed genes.")
if not shown_original["query"].isin(lfc.index).all():
    raise ValueError("Some displayed genes are missing from the LFC matrix.")
if not shown_original["display_group"].isin(target_order).all():
    raise ValueError("Some display groups are missing from the saved target order.")

print(f"{len(lfc):,} candidate genes; {len(shown_original)} displayed; {len(lfc.columns)} contrasts")
print(f"Saved stars: padj < {cfg['padj_threshold']}, |LFC| >= {cfg['lfc_threshold']}")

## Choose rows and columns (optional)

The saved subset was chosen by target-balanced absolute-LFC ranking and grouped by each gene's smallest supporting target (ties use the target order). Loading it directly preserves the original selection without repeating that analysis.

Leave this cell as-is for the original plot. You can filter `shown` or reorder `contrast_order` here. Keep each display group contiguous so its label and boundary describe one block. Reset the row index after edits, since the plot uses those positions. Filtering rows does not change the full-candidate count in the title.

In [ ]:
shown = shown_original.copy()
contrast_order = lfc.columns.tolist()

# Examples (uncomment and edit):
# shown = shown.loc[shown["display_group"].isin(["Wnt", "Wnt canonical"])].copy()
contrast_order = ["Wild", "Conventionalized", "Cvbct", "3OHC12", "EcoKD1"]

shown = shown.reset_index(drop=True)
if shown.empty:
    raise ValueError("Select at least one gene to plot.")
plot = lfc.loc[shown["query"], contrast_order]
plot_sig = sig.loc[shown["query"], contrast_order]
print(f"Plot matrix: {plot.shape[0]} rows × {plot.shape[1]} columns")

## Appearance and spacing controls

Change these values, then rerun **Draw the heatmap**. Values start at the source plot's settings (including Matplotlib's usual outer margins and colorbar geometry).

| Setting | What to adjust |
| --- | --- |
| `FIGURE_WIDTH`, `MIN_FIGURE_HEIGHT`, `ROW_HEIGHT` | Figure dimensions in inches; height is the greater of the minimum and row count × row height. |
| `MARGINS` | Axes area as fractions of the figure (0–1). Increase `left` for long group labels; increase `bottom` for rotated x labels. |
| `STRIP_WIDTH`, `HEATMAP_WIDTH` | Relative GridSpec widths for the target color strip and the heatmap. |
| `PANEL_GAP` | Gap between the strip and heatmap, as a fraction of average panel width. |
| `GROUP_TEXT_X`, `GROUP_ANCHOR_X` | Label and connector x positions in strip data coordinates. More negative moves them left. |
| `GROUP_MIN_GAP` | Minimum separation of target labels in row units; labels may move away from small blocks. Increase figure height if many labels collide. |
| `COLORBAR_FRACTION`, `COLORBAR_PAD`, `COLORBAR_SHRINK` | Colorbar width, distance from heatmap, and relative length. Its space is taken from the heatmap panel. |
| `TITLE_PAD`, `X_LABEL_PAD`, `X_TICK_PAD` | Distances in points. |
| `SHOW_GENE_LABELS` | Show symbols plus gene IDs; usually requires a taller figure and more space on the left. |
| `DISPLAY_LIMIT`, `CMAP`, `MISSING_COLOR` | Symmetric color clipping, diverging palette and missing-cell color. Statistics remain unchanged. |

For structural changes, edit the drawing cell directly: **A** creates axes, **B** draws the target strip, **C** draws values and stars, **D** places group labels, **E** adds titles and ticks. The strip's `imshow` uses integer row centers; seaborn uses centers at row index + 0.5. That difference is intentional.

In [ ]:
# Canvas and outer margins
FIGURE_WIDTH = 16
MIN_FIGURE_HEIGHT = 18
ROW_HEIGHT = 0.065
FIGURE_DPI = 600  # Inline preview; export DPI is set separately below.
MARGINS = dict(left=0.125, right=0.9, bottom=0.11, top=0.88)

# Target color strip and space between panels
STRIP_WIDTH = 0.15
HEATMAP_WIDTH = 5
PANEL_GAP = 0.07
TARGET_PALETTE = "husl"

# Group labels: x coordinates are relative to the one-column color strip
GROUP_ANCHOR_X = -0.45
GROUP_TEXT_X = -0.8
GROUP_MIN_GAP = 2
GROUP_FONT_SIZE = 12
CONNECTOR_COLOR = "grey"
CONNECTOR_WIDTH = 0.4
BOUNDARY_COLOR = "grey"
BOUNDARY_WIDTH = 0.5

# Heatmap and colorbar
DISPLAY_LIMIT = cfg["lfc_heatmap_display_limit"]
CMAP = "vlag"
MISSING_COLOR = "#bcbcbc"
COLORBAR_FRACTION = 0.15
COLORBAR_PAD = 0.05
COLORBAR_SHRINK = 1.0
COLORBAR_LABEL = "log2FoldChange"
COLORBAR_TICK_SIZE = 10

# Stars, gene labels, title, and x-axis text
SHOW_STARS = True
STAR_SIZE = 6
STAR_COLOR = "black"
SHOW_GENE_LABELS = False
GENE_LABEL_SIZE = 12
X_TICK_ROTATION = 45
X_TICK_SIZE = 10
X_TICK_ALIGNMENT = "center"
X_TICK_PAD = 3.5
X_LABEL_PAD = 4
TITLE_PAD = 6
X_LABEL = "DE source label; biological sign orientation unresolved where unverified"
TITLE = (
    f"{len(shown)} displayed / {len(lfc)} unique candidate genes; * significant, grey missing"
)

## Draw the heatmap

This is the extracted drawing code, with hard-coded appearance values exposed in the cell above. `fig`, `ax`, `axc`, and `colorbar` remain available for your own edits.

Rerunning replaces this notebook's previous figure. If you change row selection, rerun the appearance cell too so the title updates. For a new subset that changes plot size, recompute `plot` and `plot_sig` in the selection cell first.

In [ ]:
# A. Canvas and panels
if "fig" in globals():
    plt.close(fig)
fig = plt.figure(
    figsize=(FIGURE_WIDTH, max(MIN_FIGURE_HEIGHT, len(shown) * ROW_HEIGHT)),
    dpi=FIGURE_DPI,
)
fig.subplots_adjust(**MARGINS)
grid = fig.add_gridspec(1, 2, width_ratios=[STRIP_WIDTH, HEATMAP_WIDTH], wspace=PANEL_GAP)
axc = fig.add_subplot(grid[0, 0])
ax = fig.add_subplot(grid[0, 1])

# B. Target color strip; use all targets so colors match the source plot.
palette = dict(zip(target_order, sns.color_palette(TARGET_PALETTE, len(target_order))))
axc.imshow(
    np.array([palette[target] for target in shown["display_group"]])[:, None, :],
    aspect="auto", origin="upper",
)
axc.set_xticks([])
axc.set_yticks([])

# C. LFC values, missing cells, colorbar, and saved significance flags
ax.set_facecolor(MISSING_COLOR)
if SHOW_GENE_LABELS:
    gene_labels = [
        f"{row.Preferred_name} | {row.query}" if row.Preferred_name else row.query
        for row in shown.itertuples()
    ]
else:
    gene_labels = False
sns.heatmap(
    plot, mask=plot.isna(), cmap=CMAP, center=0,
    vmin=-DISPLAY_LIMIT, vmax=DISPLAY_LIMIT,
    yticklabels=gene_labels, ax=ax,
    cbar_kws=dict(
        label=COLORBAR_LABEL, fraction=COLORBAR_FRACTION,
        pad=COLORBAR_PAD, shrink=COLORBAR_SHRINK,
    ),
)

colorbar = ax.collections[0].colorbar
colorbar.ax.tick_params(axis="y", labelsize=14)
colorbar.set_label(COLORBAR_LABEL, fontsize=16)

if SHOW_GENE_LABELS:
    ax.tick_params(axis="y", labelsize=GENE_LABEL_SIZE, rotation=0)
if SHOW_STARS:
    for row, gene in enumerate(shown["query"]):
        for column, name in enumerate(plot):
            if plot_sig.loc[gene, name]:
                ax.text(
                    column + 0.5, row + 0.5, "*",
                    ha="center", va="center", fontsize=STAR_SIZE, color=STAR_COLOR,
                )

# D. Group boundaries and labels; preserve the source's label-spacing algorithm.
blocks = []
for target, block in shown.groupby("display_group", sort=False):
    first, last = block.index.min(), block.index.max()
    blocks.append((target, (first + last) / 2))
    ax.axhline(last + 1, color=BOUNDARY_COLOR, linewidth=BOUNDARY_WIDTH)

label_positions = [center for _, center in blocks]
for i in range(1, len(label_positions)):
    label_positions[i] = max(label_positions[i], label_positions[i - 1] + GROUP_MIN_GAP)
if label_positions:
    label_positions[-1] = min(label_positions[-1], len(shown) - 1)
for i in range(len(label_positions) - 2, -1, -1):
    label_positions[i] = min(label_positions[i], label_positions[i + 1] - GROUP_MIN_GAP)
for (target, center), position in zip(blocks, label_positions):
    axc.annotate(
        target, xy=(GROUP_ANCHOR_X, center), xytext=(GROUP_TEXT_X, position),
        ha="right", va="center", fontsize=GROUP_FONT_SIZE,
        arrowprops=dict(arrowstyle="-", color=CONNECTOR_COLOR, lw=CONNECTOR_WIDTH),
        annotation_clip=False,
    )

# E. Titles and axis text
ax.set_ylabel("")
#ax.set_xlabel(X_LABEL, labelpad=X_LABEL_PAD)
ax.set_title(TITLE, pad=TITLE_PAD)
ax.set_xticklabels(["Wild vs. GF", "Conv. vs. GF", "Cvbct. vs. GF", "3OHC12 vs. EtAc", "EcoKD1 vs. Ctrl."])
ax.tick_params(axis="x", rotation=X_TICK_ROTATION, labelsize=16, pad=X_TICK_PAD)
plt.setp(ax.get_xticklabels(), ha=X_TICK_ALIGNMENT)

plt.savefig("../results/figures/development_pathways/heatmap.svg", dpi=400)
#plt.show()

## Optional export

Set `SAVE_FIGURE = True` and run the next cell to write PNG and PDF versions to `results/figures/development_pathways/heatmap_refactor/`. Change `EXPORT_STEM` to keep multiple variants. Reusing the same name replaces that variant.

`bbox_inches="tight"` includes labels outside the axes and crops unused outer canvas. Set `EXPORT_BBOX = None` to keep the exact figure dimensions; in that case, keep labels inside the canvas using `MARGINS`. `EXPORT_PAD` is extra padding in inches when using a tight bounding box. Export uses the existing `fig`, so it works after the inline preview.

In [ ]:
SAVE_FIGURE = False
EXPORT_STEM = "candidate_LFC_heatmap"
EXPORT_DPI = 170
EXPORT_BBOX = "tight"
EXPORT_PAD = 0.1

if SAVE_FIGURE:
    EXPORT_DIR.mkdir(parents=True, exist_ok=True)
    for extension in ["png", "pdf"]:
        metadata = (
            {"CreationDate": None, "ModDate": None} if extension == "pdf"
            else {"Software": "HydraAEP development candidate analysis"}
        )
        destination = EXPORT_DIR / f"{EXPORT_STEM}.{extension}"
        fig.savefig(
            destination, dpi=EXPORT_DPI, bbox_inches=EXPORT_BBOX,
            pad_inches=EXPORT_PAD, metadata=metadata,
        )
        print(destination)
else:
    print("Preview only. Set SAVE_FIGURE = True to export this figure.")